# Checkout Process Optimization Analysis

## Background
ShopEase is experiencing a high shopping cart abandonment rate, resulting in significant revenue loss.

## Objective
Analyze user checkout behavior to identify factors contributing to cart abandonment and recommend improvements.

In [3]:
!pip install faker 


   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 310.6 kB/s eta 0:00:05
   ---------- ----------------------------- 0.5/2.0 MB 310.6 kB/s eta 0:00:05
   ---------- ----------------------------- 0.5/2.0 MB 310.6 kB/s eta 0:00:05
   --------------- ------------------------ 0.8/2.0 MB 364.6 kB/s eta 0:00:04
   --------------- ------------------------ 0.8/2.0 MB 364.6 kB/s eta 0:00:04
   -------------------- ------------------- 1.0/2.0 MB 405

In [4]:
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()

In [7]:
data = []

checkout_steps = [
    "Cart",
    "Shipping",
    "Payment",
    "Review",
    "Completed"
]

for i in range(1000):

    device = random.choice([
        "Mobile",
        "Desktop",
        "Tablet"
    ])

    promo = random.choice([
        "Yes",
        "No"
    ])

    step = random.choices(
        checkout_steps,
        weights=[5,10,15,20,50]
    )[0]

    abandoned = "No" if step == "Completed" else "Yes"

    data.append({
        "user_id": i + 1,
        "session_duration": random.randint(1,30),
        "cart_value": round(random.uniform(20,500),2),
        "checkout_step_reached": step,
        "abandoned": abandoned,
        "device_type": device,
        "promo_applied": promo
    })

df = pd.DataFrame(data)

df.head()


,user_id,session_duration,cart_value,checkout_step_reached,abandoned,device_type,promo_applied
0,1,9,448.73,Review,Yes,Tablet,No
1,2,26,368.15,Review,Yes,Desktop,Yes
2,3,8,246.72,Review,Yes,Tablet,Yes
3,4,5,361.84,Review,Yes,Desktop,No
4,5,3,87.98,Completed,No,Tablet,Yes


In [9]:
import sqlite3

conn = sqlite3.connect("shopease.db")

df.to_sql(
    "user_sessions",
    conn,
    if_exists="replace",
    index=False
)

1000

In [13]:
#Finding Abandonment Rate by Device
query="""SELECT device_type,COUNT(*) AS total_sessions, SUM(CASE WHEN abandoned='Yes' THEN 1 ELSE 0 END) AS abandoned_sessions, Round(100.0* SUM(CASE WHEN abandoned = 'Yes' THEN 1 ELSE 0 END)/Count(8),2) AS abandonment_rate FROM user_sessions GROUP BY device_type"""
pd.read_sql(query,conn)

,device_type,total_sessions,abandoned_sessions,abandonment_rate
0,Desktop,352,167,47.44
1,Mobile,327,163,49.85
2,Tablet,321,161,50.16


In [15]:
#Average Cart Value
quey="""SELECT abandoned,ROUND(AVG(cart_value),2) AS avg_cart_value FROM user_sessions
Group BY abandoned"""
pd.read_sql(quey,conn)

,abandoned,avg_cart_value
0,No,245.66
1,Yes,257.20


In [17]:
#Chekout Funnel
query = """
SELECT
checkout_step_reached,
COUNT(*) AS users
FROM user_sessions
GROUP BY checkout_step_reached
"""

pd.read_sql(query, conn)

,checkout_step_reached,users
0,Cart,48
1,Completed,509
2,Payment,141
3,Review,206
4,Shipping,96


In [19]:
#Promotion Impact
query = """SELECT promo_applied,
ROUND(100.0 *SUM(CASE WHEN abandoned='No'THEN 1 ELSE 0 END)/ COUNT(*),2) AS completion_rate
FROM user_sessions
GROUP BY promo_applied
"""
pd.read_sql(query, conn)

,promo_applied,completion_rate
0,No,49.41
1,Yes,52.44


In [21]:
df.to_csv("shopease_checkout_data.csv")
#Created a csv file

In [23]:
import os
print(os.getcwd())
#finding the save folder

C:\Users\Acer\CV project
